# AlphaFold DB Fetch — Example

This notebook demonstrates `run_alphafold_db_fetch`, which retrieves an AlphaFold prediction record from the [AlphaFold Protein Structure Database](https://alphafold.ebi.ac.uk/) by UniProt accession. It returns the predicted structure (PDB or mmCIF), per-residue pLDDT confidence scores, and optionally the predicted aligned error (PAE) matrix, alongside metadata such as gene name, organism, model version, and source URLs. See the AlphaFold DB paper: Varadi et al., *Nucleic Acids Research* (2022), [doi:10.1093/nar/gkab1061](https://doi.org/10.1093/nar/gkab1061).

In [1]:
from proto_tools.tools.database_retrieval import (
    AlphaFoldDBFetchConfig,
    AlphaFoldDBFetchInput,
    run_alphafold_db_fetch,
)

## Input API Reference

| Field | Type | Required | Description |
|---|---|---|---|
| `uniprot_id` | `str` | yes | UniProt accession (e.g. `'P04637'`) |

## Config API Reference

| Field | Type | Default | Description |
|---|---|---|---|
| `structure_format` | `Literal["pdb", "cif"]` | `"pdb"` | Structure file format to download |
| `include_structure` | `bool` | `True` | Download the structure file and per-residue pLDDT into `output.structure`; set False for metadata-only probes |
| `include_pae` | `bool` | `False` | Also download the PAE matrix into `output.structure.metrics["pae_matrix"]` (large for long proteins; no-op when `include_structure=False`) |
| `include_msa` | `bool` | `False` | Also download the A3M MSA into `output.msa_a3m` |

## Output API Reference

| Field | Type | Description |
|---|---|---|
| `uniprot_accession` | `str` | Primary UniProt accession that was looked up |
| `entry_id` | `str` | AlphaFold entry identifier (e.g. `'AF-P04637-F1'`) |
| `sequence` | `str` | Amino-acid sequence covered by the prediction |
| `sequence_length` | `int` | Length of the predicted sequence |
| `latest_version` | `int` | Latest available model version on AlphaFold DB |
| `mean_plddt` | `float \| None` | Mean per-residue pLDDT for the prediction (always populated; mirrored at `structure.metrics["avg_plddt"]` when a structure is fetched) |
| `pdb_url` | `str` | URL to the PDB structure file on AlphaFold DB |
| `cif_url` | `str` | URL to the mmCIF structure file on AlphaFold DB |
| `plddt_doc_url` | `str` | URL to the per-residue pLDDT JSON document |
| `pae_doc_url` | `str` | URL to the PAE JSON document |
| `structure` | `Structure \| None` | Parsed [`Structure`](../../../../entities/structures/structure.py) (PDB or mmCIF body, `b_factor_type=PLDDT`, `metrics=AlphaFoldDBMetrics` carrying `avg_plddt`, `plddt_per_residue`, and — when `include_pae=True` — `pae_matrix`). `None` when `include_structure=False`. Drop-in compatible with every structure-consuming tool in proto-tools |
| `msa_a3m` | `str \| None` | A3M-format MSA contents used as input to prediction |
| `raw_entry` | `dict[str, Any]` | Complete AlphaFold DB JSON record for advanced programmatic access |

In [2]:
# Basic usage: fetch the AlphaFold prediction for human TP53 (UniProt P04637)
# with all defaults (PDB structure + per-residue pLDDT, PAE off).
inputs = AlphaFoldDBFetchInput(uniprot_id="P04637")
config = AlphaFoldDBFetchConfig()

output = run_alphafold_db_fetch(inputs, config)

print(f"UniProt accession: {output.uniprot_accession}")
print(f"AlphaFold entry id: {output.entry_id}")
print(f"Sequence length:   {output.sequence_length}")
print(f"Mean pLDDT:        {output.mean_plddt}")

print(f"\nStructure format:  {output.structure.structure_format}")
print(f"B-factor type:     {output.structure.b_factor_type.value}")
print("\nStructure file (first 80 chars):")
print(output.structure.structure[:80])

plddt = output.structure.metrics["plddt_per_residue"]
print(f"\nstructure.metrics keys:   {sorted(output.structure.metrics.keys())}")
print(f"per-residue pLDDT length: {len(plddt)}")
print(f"first 5 pLDDT values:     {plddt[:5]}")

UniProt accession: P04637
AlphaFold entry id: AF-P04637-F1
Sequence length:   393
Mean pLDDT:        75.06

Structure format:  pdb
B-factor type:     pLDDT

Structure file (first 80 chars):
HEADER                                            01-AUG-25                     

structure.metrics keys:   ['avg_plddt', 'plddt_per_residue']
per-residue pLDDT length: 393
first 5 pLDDT values:     [42.72, 43.59, 37.78, 45.91, 39.31]


In [3]:
# With the PAE matrix: useful for assessing inter-domain confidence.
# PAE files can be large for long proteins, so this is opt-in.
pae_output = run_alphafold_db_fetch(
    AlphaFoldDBFetchInput(uniprot_id="P04637"),
    AlphaFoldDBFetchConfig(include_pae=True),
)

pae = pae_output.structure.metrics["pae_matrix"]
print(f"PAE matrix shape: ({len(pae)}, {len(pae[0])})")

print("\nTop-left 3x3 corner (angstroms):")
for row in pae[:3]:
    print([round(v, 3) for v in row[:3]])

PAE matrix shape: (393, 393)

Top-left 3x3 corner (angstroms):
[0.0, 1.0, 3.0]
[2.0, 0.0, 1.0]
[3.0, 2.0, 0.0]


In [4]:
# Metadata-only fetch: skip the structure download.
# This is the cheapest call when you only need URLs and identifiers.
meta_output = run_alphafold_db_fetch(
    AlphaFoldDBFetchInput(uniprot_id="P04637"),
    AlphaFoldDBFetchConfig(include_structure=False),
)

print(f"PDB URL:        {meta_output.pdb_url}")
print(f"mmCIF URL:      {meta_output.cif_url}")
print(f"pLDDT doc URL:  {meta_output.plddt_doc_url}")
print(f"PAE doc URL:    {meta_output.pae_doc_url}")
print(f"Mean pLDDT:     {meta_output.mean_plddt}")
print(f"structure is None: {meta_output.structure is None}")

PDB URL:        https://alphafold.ebi.ac.uk/files/AF-P04637-F1-model_v6.pdb
mmCIF URL:      https://alphafold.ebi.ac.uk/files/AF-P04637-F1-model_v6.cif
pLDDT doc URL:  https://alphafold.ebi.ac.uk/files/AF-P04637-F1-confidence_v6.json
PAE doc URL:    https://alphafold.ebi.ac.uk/files/AF-P04637-F1-predicted_aligned_error_v6.json
Mean pLDDT:     75.06
structure is None: True


In [5]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

output.export("alphafold_db_results", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'alphafold_db_results.json'}")

Exported to example_output/alphafold_db_results.json
